<a href="https://colab.research.google.com/github/garvit-07/Sentiment_analysis/blob/main/Sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q transformers datasets evaluate accelerate gradio scikit-learn seaborn

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt, seaborn as sns
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)
import evaluate

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

LABEL_NAMES = ['sadness','joy','love','anger','fear','surprise']
LABEL_EMOJI = ['😢','😊','❤️','😠','😨','😮']
NUM_LABELS  = 6
MAX_LENGTH  = 128

In [ ]:

ds = load_dataset('dair-ai/emotion')
print(ds)

fig, axes = plt.subplots(1,3, figsize=(14,4))
for ax, split in zip(axes, ['train','validation','test']):
    counts = Counter(ds[split]['label'])
    ax.bar(LABEL_NAMES, [counts[i] for i in range(6)],
           color=['#3498DB','#F1C40F','#E91E8C','#E74C3C','#8E44AD','#1ABC9C'])
    ax.set_title(f'{split} ({len(ds[split]):,} samples)')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('dair-ai/emotion Label Distribution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('\nSample tweet:', ds['train'][0])

In [ ]:

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)

tokenized = ds.map(tokenize_fn, batched=True, batch_size=512)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids','attention_mask','labels'])
print('Tokenization done ✅')

In [ ]:

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=NUM_LABELS
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:

acc_m = evaluate.load('accuracy')
f1_m  = evaluate.load('f1')

def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=-1)
    return {
        'accuracy':    acc_m.compute(predictions=preds, references=ep.label_ids)['accuracy'],
        'f1_weighted': f1_m.compute(predictions=preds, references=ep.label_ids, average='weighted')['f1'],
    }

In [ ]:

args = TrainingArguments(
    output_dir                  = '/content/emotion_model',
    num_train_epochs            = 4,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    lr_scheduler_type           = 'cosine',
    eval_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_weighted',
    fp16                        = True,   # T4 GPU supports FP16
    report_to                   = 'none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized['train'],
    eval_dataset =tokenized['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:

out    = trainer.predict(tokenized['test'])
preds  = np.argmax(out.predictions, axis=-1)
labels = out.label_ids

print(classification_report(labels, preds, target_names=LABEL_NAMES))

cm = confusion_matrix(labels, preds)
label_display = [f'{LABEL_EMOJI[i]} {LABEL_NAMES[i]}' for i in range(6)]
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=label_display, yticklabels=label_display)
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
def predict(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()

    inp = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )


    inp = {k: v.to(device) for k, v in inp.items()}

    with torch.no_grad():
        outputs = model(**inp)
        probs = torch.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()

    pred = int(np.argmax(probs))

    print(f'\n"{text}"')
    print(f'  → {LABEL_EMOJI[pred]} {LABEL_NAMES[pred].upper()} ({probs[pred]:.1%} confidence)')

predict('I just got promoted at work and I cannot stop smiling!')
predict('I miss her so much, the silence at home is unbearable.')
predict('How dare you speak to me like that, I am absolutely furious.')
predict('She surprised me with a trip to Paris this weekend!')